## 01 — Ingestion health
Lists tables, sources, freshness, and drop counters for the verification run. Asserts fixture rows exist for both venues.

In [ ]:
import clickhouse_connect, os
run_id = os.environ.get('ERGO_RUN_ID', 'latest')
client = clickhouse_connect.get_client(
    host=os.environ.get('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.environ.get('CLICKHOUSE_PORT', '8123')),
    username=os.environ.get('CLICKHOUSE_USER', 'default'),
    password=os.environ.get('CLICKHOUSE_PASSWORD', ''),
    database=os.environ.get('CLICKHOUSE_DATABASE', 'market'),
)
print('clickhouse', client.server_version)


In [ ]:
tables = client.query('SELECT name FROM system.tables WHERE database = \'market\'').result_rows
tables = [t[0] for t in tables]
print('tables:', tables)
assert 'raw_exchange_messages' in tables, 'raw capture table missing'

In [ ]:
for venue in ['binance', 'bybit']:
    n = client.query(f"SELECT count() FROM raw_exchange_messages WHERE venue = '{venue}'").result_rows[0][0]
    print(venue, 'raw rows:', n)
    assert n > 0, f'no raw rows for {venue}'

In [ ]:
for table in ['sbe_messages', 'trades', 'quotes', 'l2_books', 'funding_rates']:
    n = client.query(f'SELECT count() FROM {table} FINAL').result_rows[0][0]
    print(table, n)
    assert n > 0, f'{table} empty'